In [1]:
import pandas as pd

In [2]:
df_recommendations = pd.read_csv(
    "../data/product_relationships.csv"
)

df_recommendations.head()

,Product_A,Product_A_Name,Product_B,Product_B_Name,Purchase_Count,Confidence,Lift
0,22697,GREEN REGENCY TEACUP AND SAUCER,22699,ROSES REGENCY TEACUP AND SAUCER,428,0.773960,19.218577
1,22386,JUMBO BAG PINK POLKADOT,85099B,JUMBO BAG RED RETROSPOT,415,0.600579,7.368768
2,22726,ALARM CLOCK BAKELIKE GREEN,22727,ALARM CLOCK BAKELIKE RED,410,0.663430,15.039361
3,22697,GREEN REGENCY TEACUP AND SAUCER,22698,PINK REGENCY TEACUP AND SAUCER,368,0.665461,23.676167
4,23203,JUMBO BAG DOILEY PATTERNS,85099B,JUMBO BAG RED RETROSPOT,356,0.416862,5.114662


In [3]:
df_recommendations.shape

(265, 7)

In [4]:
df_recommendations.columns

Index(['Product_A', 'Product_A_Name', 'Product_B', 'Product_B_Name',
       'Purchase_Count', 'Confidence', 'Lift'],
      dtype='str')

In [5]:
def get_recommendations(stock_code, top_n=5):

    recommendations = df_recommendations[
        df_recommendations["Product_A"].astype(str) == str(stock_code)
    ]

    recommendations = recommendations.sort_values(
        by="Lift",
        ascending=False
    )

    return recommendations[
        [
            "Product_A",
            "Product_A_Name",
            "Product_B",
            "Product_B_Name",
            "Purchase_Count",
            "Confidence",
            "Lift"
        ]
    ].head(top_n)

In [6]:
get_recommendations("22697")

,Product_A,Product_A_Name,Product_B,Product_B_Name,Purchase_Count,Confidence,Lift
3,22697,GREEN REGENCY TEACUP AND SAUCER,22698,PINK REGENCY TEACUP AND SAUCER,368,0.665461,23.676167
0,22697,GREEN REGENCY TEACUP AND SAUCER,22699,ROSES REGENCY TEACUP AND SAUCER,428,0.773960,19.218577


In [7]:
import pandas as pd

df_products = pd.read_excel("../data/Online Retail.xlsx")

df_products.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [8]:
products = (
    df_products[["StockCode", "Description"]]
    .dropna()
    .drop_duplicates("StockCode")
    .rename(columns={
        "StockCode": "stock_code",
        "Description": "name"
    })
)

products.to_csv(
    "../data/products.csv",
    index=False
)

products.shape

(3958, 2)

In [9]:
products.head()

,stock_code,name
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER
1,71053,WHITE METAL LANTERN
2,84406B,CREAM CUPID HEARTS COAT HANGER
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE
4,84029E,RED WOOLLY HOTTIE WHITE HEART.


In [10]:
products["stock_code"].nunique()

3958

In [11]:
import os

os.path.exists("../data/products.csv")

True

In [12]:
product_records = products.to_dict("records")

len(product_records)

3958

In [13]:
from neo4j import GraphDatabase
import os
from dotenv import load_dotenv

load_dotenv()

driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(
        os.getenv("NEO4J_USERNAME"),
        os.getenv("NEO4J_PASSWORD")
    )
)

In [14]:
driver.verify_connectivity()

print("Neo4j connected!")

Neo4j connected!


In [15]:
query = """
UNWIND $products AS product

MERGE (p:Product {stock_code: product.stock_code})

SET p.name = product.name

RETURN count(p) AS products_processed
"""

In [16]:
with driver.session() as session:
    result = session.run(
        query,
        products=product_records
    )

    print(result.single()["products_processed"])

3958


In [19]:
print(products["stock_code"].duplicated().sum())
print(products["stock_code"].nunique())

0
3958


In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from src.rag import ProductRAG

In [3]:
rag = ProductRAG(
    product_file="../data/products.csv",
    persist_directory="../data/chroma_db"
)

Loaded 3958 unique products


d:\razorpay-ai-commerce-agent\src\rag.py:61: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ChromaDB loaded successfully


In [4]:
results = rag.search(
    "GREEN REGENCY TEACUP AND SAUCER",
    k=5
)

for doc in results:
    print(doc.page_content)
    print("-" * 50)

Product Name: GREEN REGENCY TEACUP AND SAUCER
Stock Code: 22697
--------------------------------------------------
Product Name: PINK REGENCY TEACUP AND SAUCER
Stock Code: 22698
--------------------------------------------------
Product Name: ROSES REGENCY TEACUP AND SAUCER 
Stock Code: 22699
--------------------------------------------------
Product Name: REGENCY TEA PLATE GREEN 
Stock Code: 23171
--------------------------------------------------
Product Name: REGENCY TEA SPOON
Stock Code: 23160
--------------------------------------------------


In [2]:
from src.agent import ask_agent

Loaded 3958 unique products


d:\razorpay-ai-commerce-agent\src\rag.py:61: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ChromaDB loaded successfully


In [3]:
print(ask_agent(
    "Find products similar to GREEN REGENCY TEACUP AND SAUCER"
))

**Products similar to “GREEN REGENCY TEACUP AND SAUCER”**

| Product Name | Stock Code |
|--------------|------------|
| GREEN REGENCY TEACUP AND SAUCER | 22697 |
| PINK REGENCY TEACUP AND SAUCER | 22698 |
| ROSES REGENCY TEACUP AND SAUCER | 22699 |
| REGENCY TEA PLATE GREEN | 23171 |
| REGENCY TEA SPOON | 23160 |

These items were retrieved via semantic similarity search and match the style, brand, or theme of the green Regency set.


In [4]:
print(ask_agent(
    "Which of those products is pink?"
))

The pink item is:

- **PINK REGENCY TEACUP AND SAUCER** – Stock Code **22698**


In [6]:
response = ask_agent(
    "I bought product 22697. What products should I buy with it?"
)

print(response)

Here are the two items that most customers add to their cart after buying **product 22697**:

| Stock Code | Product Name | Purchase Count | Confidence | Lift |
|------------|--------------|----------------|------------|------|
| **22698** | PINK REGENCY TEACUP AND SAUCER | 368 | 0.665 | 23.68 |
| **22699** | ROSES REGENCY TEACUP AND SAUCER | 428 | 0.774 | 19.22 |

**Why these?**  
- Both are part of the same “Regency” tea set line, so they naturally complement the item you already bought.  
- The high lift values (23.68 and 19.22) indicate that customers who buy 22697 are much more likely to buy these teacups than the average shopper.  

If you’re looking to complete a tea set or add a matching saucer, either of these would be a great choice!


In [7]:
response = ask_agent(
    "Find products similar to GREEN REGENCY TEACUP AND SAUCER"
)

print(response)

Here are a few products that are similar to the **GREEN REGENCY TEACUP AND SAUCER**:

| # | Product Name | Stock Code | Why it’s similar |
|---|--------------|------------|------------------|
| 1 | **PINK REGENCY TEACUP AND SAUCER** | 22698 | Same “Regency” design line, just a different color. |
| 2 | **ROSES REGENCY TEACUP AND SAUCER** | 22699 | Same style and size, floral‑pink theme. |
| 3 | **REGENCY TEA PLATE GREEN** | 23171 | Matches the green Regency theme; great for pairing with the cup set. |
| 4 | **REGENCY TEA SPOON** | 23160 | Complements the cup & saucer set; same Regency style. |

If you’d like to explore more items in the Regency collection or need additional recommendations, just let me know!
